# Setup

In [1]:
!nvidia-smi

Sun Mar 23 18:06:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
!pip install pandas
!pip install seaborn
!pip install numpy
!pip install scikit-learn
!pip install matplotlib
!pip install scipy
!pip install torch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Any
from pandas.core.frame import DataFrame
from pandas import Series
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, PowerTransformer, StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import os
from datetime import timedelta

In [4]:
data_dir = "/kaggle/input/dataflow-final"
stock_paths = {
    file.split('_')[-1].replace('.csv', ''): os.path.join(data_dir, file)
    for file in os.listdir(data_dir)
    if file.endswith('.csv')
}

stock_paths

{'CTS': '/kaggle/input/dataflow-final/final_CTS.csv',
 'PVC': '/kaggle/input/dataflow-final/final_PVC.csv',
 'MSN': '/kaggle/input/dataflow-final/final_MSN.csv',
 'VNM': '/kaggle/input/dataflow-final/final_VNM.csv',
 'CMG': '/kaggle/input/dataflow-final/final_CMG.csv',
 'NLG': '/kaggle/input/dataflow-final/final_NLG.csv',
 'PDR': '/kaggle/input/dataflow-final/final_PDR.csv',
 'MWG': '/kaggle/input/dataflow-final/final_MWG.csv',
 'HAG': '/kaggle/input/dataflow-final/final_HAG.csv',
 'PVD': '/kaggle/input/dataflow-final/final_PVD.csv',
 'KDH': '/kaggle/input/dataflow-final/final_KDH.csv',
 'CTG': '/kaggle/input/dataflow-final/final_CTG.csv',
 'PVS': '/kaggle/input/dataflow-final/final_PVS.csv',
 'VIC': '/kaggle/input/dataflow-final/final_VIC.csv',
 'POT': '/kaggle/input/dataflow-final/final_POT.csv',
 'VCB': '/kaggle/input/dataflow-final/final_VCB.csv',
 'NVB': '/kaggle/input/dataflow-final/final_NVB.csv',
 'FPT': '/kaggle/input/dataflow-final/final_FPT.csv',
 'PVB': '/kaggle/input/dataf

In [5]:
def clean_data(df: DataFrame) -> DataFrame:
    remove_features: list[str] = ['symbol', 'open_kVND', 'high_kVND', 'low_kVND']
    df: DataFrame = df.drop(columns=remove_features, errors='ignore')

    return df

stock_dfs = {
    stock_name: clean_data(pd.read_csv(stock_path)) for stock_name, stock_path in stock_paths.items()
}

In [6]:
len(stock_dfs['CTS'].columns.tolist())

201

In [7]:
for stock_name, df in stock_dfs.items():
    df['price_shift_60'] = df['close_kVND'].shift(-60) # adjust to 60
    df['return_60'] = (df['price_shift_60'] - df['close_kVND'])*100/df['close_kVND']
    stock_dfs[stock_name] = df.dropna(subset=['price_shift_60'], axis=0)

In [8]:
for stock_name, df in stock_dfs.items():
  if not isinstance(df.index, pd.DatetimeIndex):
    if 'time' in df.columns:
      df['time'] = pd.to_datetime(df['time'])
      df.set_index('time', inplace=True)

# Using p-value and correlation

In [9]:
selected_features_per_stock = {}

for stock_name, df in stock_dfs.items():
    features = df.drop(columns=['price_shift_60']).columns
    label = df['price_shift_60']
    
    selected_features = set()
    
    for feature in features:
        # Handle potential NaNs or constant columns
        if df[feature].isna().any() or df[feature].std() == 0:
            continue

        corr, p_value = pearsonr(df[feature], label)
        if abs(corr) > 0.1 and p_value < 0.05:
            selected_features.add(feature)
    
    print(f"{stock_name}: Selected {len(selected_features)} features out of {len(features)}")
    selected_features_per_stock[stock_name] = selected_features

shared_selected_features = set.intersection(*selected_features_per_stock.values())
print(f"Shared features across all stocks: {len(shared_selected_features)} features")
for stock_name in stock_dfs.keys():
    stock_dfs[stock_name] = stock_dfs[stock_name][list(shared_selected_features) + ['price_shift_60']]

CTS: Selected 136 features out of 201
PVC: Selected 122 features out of 201
MSN: Selected 121 features out of 201
VNM: Selected 131 features out of 201
CMG: Selected 119 features out of 201
NLG: Selected 125 features out of 201
PDR: Selected 120 features out of 201
MWG: Selected 127 features out of 201
HAG: Selected 133 features out of 201
PVD: Selected 126 features out of 201
KDH: Selected 122 features out of 201


<ipython-input-9-fe032f39efac>:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_value = pearsonr(df[feature], label)


CTG: Selected 135 features out of 201
PVS: Selected 119 features out of 201
VIC: Selected 121 features out of 201
POT: Selected 129 features out of 201


<ipython-input-9-fe032f39efac>:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_value = pearsonr(df[feature], label)


VCB: Selected 123 features out of 201
NVB: Selected 112 features out of 201
FPT: Selected 123 features out of 201


<ipython-input-9-fe032f39efac>:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_value = pearsonr(df[feature], label)


PVB: Selected 140 features out of 201
ITD: Selected 125 features out of 201
Shared features across all stocks: 81 features


In [10]:
from sklearn.pipeline import Pipeline

def create_train_df_per_stock(features: np.array, label: np.array, lookback: int) -> tuple:
    X, y = [], []
    for i in range(lookback, len(features)):  
        X.append(features[i-lookback:i])  
        y.append(label[i])
    return np.array(X), np.array(y)


label_scalers = {
    stock_name: Pipeline([
        ('minmax_scaler', MinMaxScaler()),
        ('power_transformer', PowerTransformer())
    ]) for stock_name in stock_dfs
}

feature_scalers = {
    stock_name: StandardScaler() for stock_name in stock_dfs
}

datasets = {
    stock_name: {"X_train": None, "X_test": None, "y_train": None, "y_test": None} for stock_name in stock_dfs
}

for stock_name, df in stock_dfs.items():
    lookback = 150

    X = df.drop(["price_shift_60"], axis=1)
    y = df['price_shift_60'].values

    y = label_scalers[stock_name].fit_transform(y.reshape(-1, 1))
    X = feature_scalers[stock_name].fit_transform(X) 
    X, y = create_train_df_per_stock(X, y, lookback) # X shape (2347, 150, 84) | y shape (2347,)

    split = int(len(y) * 0.85)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Store the datasets for each stock
    datasets[stock_name]["X_train"] = X_train
    datasets[stock_name]["X_test"] = X_test
    datasets[stock_name]["y_train"] = y_train
    datasets[stock_name]["y_test"] = y_test
    print(X_train.shape, X_test.shape)

(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)
(2021, 150, 81) (357, 150, 81)


In [11]:
total_len_train = 0
for stock_name, dataset in datasets.items():
  if dataset["X_train"] is not None:
    total_len_train += len(dataset["X_train"])

print(f"Total length of X_train across all stocks: {total_len_train}")

Total length of X_train across all stocks: 40420


# XGBoost Finetuning

# LSTM Finetuning

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import numpy as np

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 64

Using device: cuda


In [13]:
X_train = np.concatenate([d["X_train"] for d in datasets.values() if d["X_train"] is not None])
y_train = np.concatenate([d["y_train"] for d in datasets.values() if d["y_train"] is not None])
X_test = np.concatenate([d["X_test"] for d in datasets.values() if d["X_test"] is not None])
y_test = np.concatenate([d["y_test"] for d in datasets.values() if d["y_test"] is not None])

print("Combined X_train shape:", X_train.shape)
print("Combined y_train shape:", y_train.shape)
print(X_test.shape, y_test.shape)

# Prepare data loaders
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_split = int(0.8 * len(train_dataset))
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [val_split, len(train_dataset) - val_split])
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

Combined X_train shape: (40420, 150, 81)
Combined y_train shape: (40420, 1)
(7140, 150, 81) (7140, 1)


In [14]:
def train_model(model, train_loader, val_loader, epochs=50, patience=10):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)  # reduced learning rate

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)
        print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load('best_model.pt'))
    return model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, 120, batch_first=True)
        self.lstm2 = nn.LSTM(120, 64, batch_first=True)
        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x, _ = self.lstm2(x)
        x, (hn, _) = self.lstm3(x)
        x = hn.squeeze(0)
        return self.fc(x)

# Train LSTM
print("\nTraining LSTM model:")
lstm_model = LSTMModel(X_train.shape[2]).to(device)
lstm_model = train_model(lstm_model, train_loader, val_loader)

lstm_model.eval()
with torch.no_grad():
    lstm_preds = lstm_model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

print("LSTM RMSE:", np.sqrt(mean_squared_error(y_test, lstm_preds)))
print("LSTM MAPE:", mean_absolute_percentage_error(y_test, lstm_preds))
print("LSTM R2:", r2_score(y_test, lstm_preds))


Training LSTM model:
Epoch 1, Val Loss: 0.0436
Epoch 2, Val Loss: 0.0346
Epoch 3, Val Loss: 0.0283
Epoch 4, Val Loss: 0.0262
Epoch 5, Val Loss: 0.0204
Epoch 6, Val Loss: 0.0209


In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

# LSTM Prediction

In [ ]:
lookback = 150

predictions = {}
metrics = {}

for stock_name, df in stock_dfs.items():
    df = df.sort_index()
    
    df = df[list(shared_selected_features) + ['price_shift_60']]
    
    split = int(len(df) * 0.85)
    test_df = df.iloc[split:]
    
    extended_test_df = df.iloc[split - lookback:]
    
    if len(extended_test_df) < lookback:
        print(f"Not enough test data for stock {stock_name}")
        continue
    
    X_raw = extended_test_df.drop("price_shift_60", axis=1)
    y_raw = extended_test_df["price_shift_60"]
    
    X_scaled = feature_scalers[stock_name].transform(X_raw)
    y_scaled = label_scalers[stock_name].transform(y_raw.values.reshape(-1, 1))
    X_test_all, y_test_all = create_train_df_per_stock(X_scaled, y_scaled, lookback)
    window_dates = extended_test_df.index[lookback:]
    
    mask = window_dates.isin(test_df.index)
    X_test = X_test_all[mask]
    y_test = y_test_all[mask]
    dates_selected = window_dates[mask]
    
    if X_test.ndim == 1:
        X_test = X_test.reshape(1, lookback, -1)
    elif X_test.ndim == 2:
        X_test = np.expand_dims(X_test, axis=0)
    
    lstm_model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_pred_scaled = lstm_model(X_tensor).cpu().numpy()
    
    y_pred = label_scalers[stock_name].inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = label_scalers[stock_name].inverse_transform(y_test.reshape(-1, 1)).flatten()
    
    results_df = pd.DataFrame({
        "Date": dates_selected,
        "Predicted": y_pred,
        "Actual": y_true
    })
    results_df.set_index("Date", inplace=True)
    
    predictions[stock_name] = results_df
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    metrics[stock_name] = {"RMSE": rmse, "MAPE": mape, "R2": r2}

for stock, df_results in predictions.items():
    print(f"Predictions for {stock}:")
    print(df_results)
    print("\n")

print("Performance Metrics:")
for stock, m in metrics.items():
    print(f"{stock}: RMSE={m['RMSE']:.4f}, MAPE={m['MAPE']:.4f}, R2={m['R2']:.4f}")

In [ ]:
all_predictions = pd.concat(
    [df.assign(Stock=stock) for stock, df in predictions.items()]
)

all_predictions = all_predictions.reset_index().rename(columns={'index': 'Date'})
all_predictions.head()

In [ ]:
predicted_prices_df = all_predictions.pivot(index='Date', columns='Stock', values='Predicted')
predicted_prices_df.head()

In [ ]:
len(predicted_prices_df)

In [ ]:
predicted_prices_df.to_csv('Predicted_LSTM_Prices.csv')
print("Reshaped predicted prices saved to 'Predicted_LSTM_Prices.csv'")

In [ ]:
actual_prices_df = all_predictions.pivot(index='Date', columns='Stock', values='Actual')
actual_prices_df.head()

In [ ]:
actual_prices_df.to_csv('Actual_LSTM_Prices.csv')
print("Reshaped actual prices saved to 'Actual_LSTM_Prices.csv'")

In [ ]:
# target_date = pd.to_datetime("2024-12-02")
# lookback = 150

# results = {}

# for stock_name, df in stock_dfs.items():
#     df = df.sort_index()
#     df = df[list(shared_selected_features) + ['price_shift_60']]
    
#     if target_date not in df.index:
#         print(f"Stock {stock_name}: Target date {target_date.strftime('%d/%m/%Y')} is missing.")
#         continue
    
#     df_subset = df.loc[:target_date]
    
#     if len(df_subset) < (lookback + 1):
#         print(f"Stock {stock_name}: Not enough data to form a sliding window ending on {target_date.strftime('%d/%m/%Y')}.")
#         continue
    
#     X_raw = df_subset.drop("price_shift_60", axis=1)
#     y_raw = df_subset["price_shift_60"]
    
#     X_scaled = feature_scalers[stock_name].transform(X_raw)
#     y_scaled = label_scalers[stock_name].transform(y_raw.values.reshape(-1, 1))
#     X_all, y_all = create_train_df_per_stock(X_scaled, y_scaled, lookback)
#     window_dates = df_subset.index[lookback:]
    
#     idx = np.where(window_dates == target_date)[0]
#     if len(idx) == 0:
#         print(f"Stock {stock_name}: No sliding window ends exactly on {target_date.strftime('%d/%m/%Y')}.")
#         continue
    
#     X_target = X_all[idx[0]]

#     if X_target.ndim == 2:
#         X_target = np.expand_dims(X_target, axis=0)
    
#     lstm_model.eval()
#     with torch.no_grad():
#         X_tensor = torch.tensor(X_target, dtype=torch.float32).to(device)
#         y_pred_scaled = lstm_model(X_tensor).cpu().numpy()
    
#     y_pred = label_scalers[stock_name].inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()[0]
    
#     actual_value = df_subset.loc[target_date, "price_shift_60"]
    
#     results[stock_name] = {"Predicted": y_pred, "Actual": actual_value}

# print("Prediction vs Actual for 12/02/2024:")
# for stock, comp in results.items():
#     print(f"{stock}: Predicted = {comp['Predicted']:.4f}, Actual = {comp['Actual']}")

# GRU Finetuning

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, 64, batch_first=True)
        self.gru2 = nn.GRU(64, 32, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x, _ = self.gru(x)
        x, hn = self.gru2(x)  # fixed unpacking
        x = hn.squeeze(0)
        return self.fc(x)

print("\nTraining GRU model:")
gru_model = GRUModel(X_train.shape[2]).to(device)
gru_model = train_model(gru_model, train_loader, val_loader)

# Evaluate GRU
gru_model.eval()
with torch.no_grad():
    gru_preds = gru_model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

print("GRU RMSE:", np.sqrt(mean_squared_error(y_test, gru_preds)))
print("GRU MAPE:", mean_absolute_percentage_error(y_test, gru_preds))
print("GRU R2:", r2_score(y_test, gru_preds))

# Plotting MAPE, MSE, R^2

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, stock_name):
    model.eval()
    with torch.no_grad():
        y_pred = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

    y_test_true = label_scalers[stock_name].inverse_transform(y_test.reshape(-1, 1))
    y_pred = label_scalers[stock_name].inverse_transform(y_pred.reshape(-1, 1))

    mape = mean_absolute_percentage_error(y_test_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_true, y_pred))
    r2 = r2_score(y_test_true, y_pred)

    return mape, rmse, r2


metrics_values = []
for stock_name, dataset in datasets.items():
    if dataset["X_test"] is not None:
        print(f"Evaluating for stock: {stock_name}")

        # Evaluate LSTM model and append metrics
        mape_lstm, rmse_lstm, r2_lstm = evaluate_model(lstm_model, dataset["X_test"], dataset["y_test"], "LSTM", stock_name)
        metrics_values.append((stock_name, "LSTM", mape_lstm, rmse_lstm, r2_lstm))

        # Evaluate GRU model and append metrics
        mape_gru, rmse_gru, r2_gru = evaluate_model(gru_model, dataset["X_test"], dataset["y_test"], "GRU", stock_name)
        metrics_values.append((stock_name, "GRU", mape_gru, rmse_gru, r2_gru))

metrics_df = pd.DataFrame(metrics_values, columns=['Stock', 'Model', 'MAPE', 'RMSE', 'R2'])

# Plotting the metrics for all stocks and models (bar plots)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RMSE Plot
sns.barplot(x='Stock', y='RMSE', hue='Model', data=metrics_df, ax=axes[0])
axes[0].set_title('RMSE for All Stocks and Models')
axes[0].set_ylabel('RMSE')
axes[0].set_xlabel('Stock')
axes[0].tick_params(axis='x', rotation=90)

# MAPE Plot
sns.barplot(x='Stock', y='MAPE', hue='Model', data=metrics_df, ax=axes[1])
axes[1].set_title('MAPE for All Stocks and Models')
axes[1].set_ylabel('MAPE')
axes[1].set_xlabel('Stock')
axes[1].tick_params(axis='x', rotation=90)

# R2 Plot
sns.barplot(x='Stock', y='R2', hue='Model', data=metrics_df, ax=axes[2])
axes[2].set_title('R2 for All Stocks and Models')
axes[2].set_ylabel('R2')
axes[2].set_xlabel('Stock')
axes[2].tick_params(axis='x', rotation=90)

# Adjust layout and show the plots
plt.tight_layout()
plt.show()

# Distribution of Losses (RMSE, MAPE, R2)

# Initialize the figure for distribution plots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RMSE Distribution Plot
sns.histplot(metrics_df['RMSE'], kde=True, bins=10, ax=axes[0], color='blue')
axes[0].set_title('RMSE Distribution')
axes[0].set_xlabel('RMSE')
axes[0].set_ylabel('Frequency')

# MAPE Distribution Plot
sns.histplot(metrics_df['MAPE'], kde=True, bins=10, ax=axes[1], color='green')
axes[1].set_title('MAPE Distribution')
axes[1].set_xlabel('MAPE')
axes[1].set_ylabel('Frequency')

# R2 Distribution Plot
sns.histplot(metrics_df['R2'], kde=True, bins=10, ax=axes[2], color='red')
axes[2].set_title('R2 Distribution')
axes[2].set_xlabel('R2')
axes[2].set_ylabel('Frequency')

# Adjust layout and show the distribution plots
plt.tight_layout()
plt.show()

## Flow tiếp theo
- phân tích các outliner trong các biểu đồ của hàm loss -> tại sao nó lại bị outliner
- chia nhóm cổ phiếu -> run lại model với từng nhóm cổ phiếu (chia dựa trên insight)
- feature engineering (2 cách) -> lọc ra những feature trùng nhau sau khi lọc cho từng mã -> run lại model lần nữa